# Eclipse Lab · From a date to the shadow on Earth

On 8 April 2024, observers in Dallas and San José saw the same Moon pass in front of the same Sun. Why did only one city experience totality? A date lets us ask whether there was an eclipse somewhere on Earth. Answering the question for a city requires putting the observer on the surface and finding which part of the Sun remained visible there.

Run this notebook from its own writable folder as `Eclipse_Lab_Integrated.ipynb` with Python 3.10 or later. The training table is embedded. First-time queries for ephemerides, reference data and map tiles require internet access. The final cells write and launch the local dashboard.

In [1]:
%pip install numpy pandas scipy scikit-learn requests joblib astropy folium shapely contourpy streamlit

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Lengths, times and the reference surface

We use km for positions and radii, seconds for elapsed time, and km/s for velocities. The Sun and Moon are spheres. Earth has distinct equatorial and polar radii, which matter both for where a ray meets the surface and for the direction of the local horizon.

In [2]:
from pathlib import Path
from datetime import date
from urllib.parse import urlencode, urljoin
from urllib.request import urlopen
from functools import lru_cache
from html import escape
import base64
import csv
import hashlib
import html
import io
import json
import re
import zlib
import numpy as np
import pandas as pd
import requests
import joblib
import sklearn
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix
from scipy.interpolate import CubicHermiteSpline
from astropy.time import Time, TimeDelta
from astropy.coordinates import GCRS, ITRS, CartesianRepresentation, EarthLocation
from astropy.utils import iers
import astropy.units as u
import folium
import contourpy
from shapely.geometry import Polygon, box, mapping, shape
from shapely.affinity import translate
from shapely.ops import unary_union

PROJECT_DIR = Path.cwd() / "eclipse_lab"
CACHE_DIR = PROJECT_DIR / "cache" / "physics_v1"
MODEL_DIR = PROJECT_DIR / "models"
for folder in [CACHE_DIR, MODEL_DIR, PROJECT_DIR / "assets"]:
    folder.mkdir(parents=True, exist_ok=True)
PIPELINE_VERSION = "integrated-1"
LIGHT_SPEED_KM_S = 299792.458
SUN_RADIUS_KM = 695700.0
MOON_RADIUS_KM = 1737.4
EQUATORIAL_RADIUS_KM = 6378.137
FLATTENING = 1 / 298.257223563
POLAR_RADIUS_KM = EQUATORIAL_RADIUS_KM * (1 - FLATTENING)
MONTHS = [
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun",
    "Jul",
    "Aug",
    "Sep",
    "Oct",
    "Nov",
    "Dec",
]
iers.conf.auto_download = True
POSITION_COLUMNS = [
    f"{axis}_km_{body}" for body in ["sun", "moon"] for axis in ["x", "y", "z"]
]
FEATURES = ["day_number"] + POSITION_COLUMNS + ["angle_deg"]

The 789 labeled days from 1991–2050 are stored in the notebook metadata. Reading that table from the notebook itself keeps the fitted model reproducible without a second data download.

In [11]:
FOLDER_PATH = Path(r"C:\Users\andre\Desktop\Eclipse_Project")

candidates = sorted(FOLDER_PATH.glob("Step7_Eclipse_Lab_Integrated.ipynb"))

if len(candidates) != 1:
    raise FileNotFoundError("There must be exactly one integrated notebook in the folder.")

NOTEBOOK_PATH = candidates[0]

notebook_document = json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))

TRAINING_DATA = notebook_document["metadata"]["eclipse_training_data"]

Repeated searches can reuse a result when the settings that produced it agree. The key below hashes an ordered serialization of those settings. The following helper writes the result through a temporary file.

In [12]:
def cache_key(settings):
    text = json.dumps(settings, sort_keys=True)
    return hashlib.sha256(text.encode()).hexdigest()[:24]

In [13]:
def save_json(path, data):
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(data, allow_nan=False), encoding="utf-8")
    temporary.replace(path)

## Which midnight?

The input denotes a UT1 day in the proleptic Gregorian calendar. Astronomical year 0 is 1 BCE. The accepted range is −1999 through 3000. For a historical catalog entry, first convert its calendar and the time of greatest eclipse to the corresponding observation day.

In [14]:
def date_parts(date_string):
    match = re.fullmatch("(-?\\d{1,4})-(\\d{2})-(\\d{2})", str(date_string).strip())
    if match is None:
        raise ValueError(
            "Use YYYY-MM-DD, with an astronomical year from -1999 to 3000."
        )
    year, month, day = map(int, match.groups())
    if not -1999 <= year <= 3000:
        raise ValueError("The supported year range is -1999 to 3000.")
    leap = year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)
    month_lengths = [31, 29 if leap else 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    if not 1 <= month <= 12 or not 1 <= day <= month_lengths[month - 1]:
        raise ValueError("Invalid calendar date.")
    return (year, month, day)

In [15]:
def normalize_date(date_string):
    year, month, day = date_parts(date_string)
    return (
        f"-{abs(year):04d}" if year < 0 else f"{year:04d}"
    ) + f"-{month:02d}-{day:02d}"

A Julian day number labels a day by an integer. Julian dates change integer at noon, so midnight at the beginning of that date has JD equal to JDN minus one half. We also need the inverse conversion when the path extends onto an adjacent day.

In [16]:
def date_to_jdn(date_string):
    year, month, day = date_parts(date_string)
    a = (14 - month) // 12
    y = year + 4800 - a
    m = month + 12 * a - 3
    return day + (153 * m + 2) // 5 + 365 * y + y // 4 - y // 100 + y // 400 - 32045

In [17]:
def julian_day(date_string):
    return date_to_jdn(date_string) - 0.5

In [18]:
def date_from_jd(jd):
    a = int(np.floor(jd + 0.5)) + 32044
    b = (4 * a + 3) // 146097
    c = a - 146097 * b // 4
    d = (4 * c + 3) // 1461
    e = c - 1461 * d // 4
    m = (5 * e + 2) // 153
    day = e - (153 * m + 2) // 5 + 1
    month = m + 3 - 12 * (m // 10)
    year = 100 * b + d - 4800 + m // 10
    return (
        f"-{abs(year):04d}" if year < 0 else f"{year:04d}"
    ) + f"-{month:02d}-{day:02d}"

In [19]:
def start_ut1_day(date_string, delta_t):
    return julian_day(date_string)

## Can the date produce an eclipse?

Each training row contains the day number, mean Cartesian position of each body in km, minimum observed separation in degrees, and the catalog label. The compressed representation expands to an ordinary CSV table.

In [20]:
def load_training_data():
    csv_bytes = zlib.decompress(base64.b64decode(TRAINING_DATA))
    return pd.read_csv(io.BytesIO(csv_bytes))

We split the sorted dates before fitting. The confusion matrix comes from predicting the last fifth with a model trained only on earlier dates. After measuring that performance we refit on all available days for the dashboard. A compatible saved fit can be loaded on the next run.

In [21]:
def train_model(daily):
    daily = daily.sort_values("day_number").dropna(subset=FEATURES + ["eclipse"])
    if len(daily) < 20 or set(daily.eclipse.unique()) != {0, 1}:
        raise ValueError("Training requires at least 20 days and both target classes.")
    split = int(len(daily) * 0.8)
    model = HistGradientBoostingClassifier(
        max_iter=200, learning_rate=0.05, random_state=42
    )
    model.fit(daily.iloc[:split][FEATURES], daily.iloc[:split].eclipse.astype(int))
    predicted = model.predict(daily.iloc[split:][FEATURES])
    report = classification_report(
        daily.iloc[split:].eclipse, predicted, zero_division=0, output_dict=True
    )
    matrix = confusion_matrix(
        daily.iloc[split:].eclipse, predicted, labels=[0, 1]
    ).tolist()
    model.fit(daily[FEATURES], daily.eclipse.astype(int))
    model_id = cache_key(
        {
            "data": hashlib.sha256(daily.to_csv(index=False).encode()).hexdigest(),
            "sklearn": sklearn.__version__,
            "version": PIPELINE_VERSION,
        }
    )
    bundle = {
        "model": model,
        "model_id": model_id,
        "report": report,
        "confusion_matrix": matrix,
        "first_day": int(daily.day_number.min()),
        "last_day": int(daily.day_number.max()),
        "features": FEATURES,
        "sklearn_version": sklearn.__version__,
    }
    joblib.dump(bundle, MODEL_DIR / "model_1.joblib")
    return bundle

In [22]:
def load_model():
    path = MODEL_DIR / "model_1.joblib"
    if path.exists():
        bundle = joblib.load(path)
        if bundle["sklearn_version"] == sklearn.__version__:
            return bundle
    return train_model(load_training_data())

For an unknown date there is no known hour of greatest eclipse. We sample geocentric positions at 96 quarter-hour instants. The training means came from irregular windows around selected events and controls. Averaging a full day changes the inputs even with the same six coordinate names.

In [23]:
def get_classifier_positions(body_id, day_number):
    settings = {
        "body": str(body_id),
        "day": int(day_number),
        "step": "15m",
        "center": "500@399",
        "frame": "ICRF",
        "plane": "FRAME",
        "time": "UT",
        "correction": "NONE",
    }
    path = CACHE_DIR / ("classifier_vectors_" + cache_key(settings) + ".npy")
    if path.exists():
        positions = np.load(path, allow_pickle=False)
        if positions.shape == (96, 3) and np.isfinite(positions).all():
            return positions
    start = day_number - 0.5
    parameters = {
        "format": "text",
        "COMMAND": f"'{body_id}'",
        "OBJ_DATA": "'NO'",
        "MAKE_EPHEM": "'YES'",
        "EPHEM_TYPE": "'VECTORS'",
        "CENTER": "'500@399'",
        "REF_PLANE": "'FRAME'",
        "REF_SYSTEM": "'ICRF'",
        "VEC_CORR": "'NONE'",
        "OUT_UNITS": "'KM-S'",
        "VEC_TABLE": "'1'",
        "VEC_LABELS": "'NO'",
        "CSV_FORMAT": "'YES'",
        "TIME_TYPE": "'UT'",
        "START_TIME": f"'JD{start:.8f}'",
        "STOP_TIME": f"'JD{start + 23.75 / 24:.8f}'",
        "STEP_SIZE": "'15m'",
    }
    response = requests.get(
        "https://ssd.jpl.nasa.gov/api/horizons.api", params=parameters, timeout=90
    )
    response.raise_for_status()
    if "$$SOE" not in response.text or "$$EOE" not in response.text:
        raise ValueError(
            "Horizons did not return classifier vectors: " + response.text[:300]
        )
    rows = csv.reader(
        response.text.split("$$SOE")[1].split("$$EOE")[0].strip().splitlines()
    )
    positions = np.array([[float(value) for value in row[2:5]] for row in rows])
    if positions.shape != (96, 3) or not np.isfinite(positions).all():
        raise ValueError("Expected 96 finite positions per body.")
    np.save(path, positions)
    return positions

For each instant, $\cos\alpha=(\mathbf r_S\cdot\mathbf r_M)/(|\mathbf r_S|\,|\mathbf r_M|)$. We use the smallest sampled $\alpha$ together with the mean positions and day number. A positive decision sends the date to the geometry calculation. A missed eclipse ends the search here. The result records when the date is outside the training interval.

In [24]:
def predict_eclipse(date_string, bundle=None):
    date_string = normalize_date(date_string)
    if bundle is None:
        bundle = load_model()
    key = cache_key(
        {"date": date_string, "model": bundle["model_id"], "version": PIPELINE_VERSION}
    )
    path = CACHE_DIR / ("prediction_" + key + ".json")
    if path.exists():
        return json.loads(path.read_text())
    day = date_to_jdn(date_string)
    sun = get_classifier_positions("10", day)
    moon = get_classifier_positions("301", day)
    norms = np.linalg.norm(sun, axis=1) * np.linalg.norm(moon, axis=1)
    if np.any(norms <= 0):
        raise ValueError("Invalid geocentric position vector.")
    angles = np.degrees(np.arccos(np.clip(np.sum(sun * moon, axis=1) / norms, -1, 1)))
    values = (
        [day]
        + sun.mean(axis=0).tolist()
        + moon.mean(axis=0).tolist()
        + [float(angles.min())]
    )
    frame = pd.DataFrame([values], columns=FEATURES)
    model = bundle["model"]
    class_index = list(model.classes_).index(1)
    result = {
        "date": date_string,
        "prediction": bool(model.predict(frame)[0]),
        "probability": float(model.predict_proba(frame)[0, class_index]),
        "model_id": bundle["model_id"],
        "features": dict(zip(FEATURES, values)),
        "outside_training_range": not bundle["first_day"] <= day <= bundle["last_day"],
    }
    save_json(path, result)
    return result

Read the matrix with catalog labels on the rows and predictions on the columns, in the order 0, 1. Its scope is the held-out sample drawn using the original observing windows.

In [25]:
model_bundle = load_model()
print("Chronological holdout confusion matrix [0, 1]:")
print(np.array(model_bundle["confusion_matrix"]))
display(pd.DataFrame(model_bundle["report"]).T)

Chronological holdout confusion matrix [0, 1]:
[[131   0]
 [  0  27]]


,precision,recall,f1-score,support
0,1.0,1.0,1.0,131.0
1,1.0,1.0,1.0,27.0
accuracy,1.0,1.0,1.0,1.0
macro avg,1.0,1.0,1.0,158.0
weighted avg,1.0,1.0,1.0,158.0


## Moving with Earth

Let $B(t)$ contain the Earth-fixed basis vectors expressed in celestial coordinates. An observer fixed on Earth uses $\mathbf r_T=B^\top\mathbf r$. Differentiating gives $\mathbf v_T=B^\top\mathbf v+\dot B^\top\mathbf r$. Omitting the second term would give a velocity inconsistent with the rotating position coordinates.

In [26]:
def earth_basis(t):
    itrs_axes = ITRS(CartesianRepresentation(np.eye(3) * u.km), obstime=t)
    gcrs_axes = itrs_axes.transform_to(GCRS(obstime=t))
    return gcrs_axes.cartesian.xyz.to_value(u.km)

In [27]:
def rotate_state(t, r, v, h=1.0):
    B = earth_basis(t)
    delta = TimeDelta(h, format="sec")
    B_derivative = (earth_basis(t + delta) - earth_basis(t - delta)) / (2 * h)
    r = np.asarray(r, dtype=float)
    v = np.asarray(v, dtype=float)
    r_itrs = B.T @ r
    v_itrs = B.T @ v + B_derivative.T @ r
    return (r_itrs, v_itrs)

From the Moon trace $P(\lambda)=\mathbf r_M+\lambda\hat{\mathbf d}$, where $\hat{\mathbf d}$ points away from the Sun. Substituting this ray into $(x^2+y^2)/a^2+z^2/b^2=1$ yields a quadratic. Its first forward root is the central point. If the Moon looks larger than the Sun there, the eclipse is total. If it looks smaller, the central observer sees an annulus. An axis that misses the surface can still produce a partial eclipse.

In [28]:
def central_point(rS, rL):
    rS = np.asarray(rS, dtype=float)
    rL = np.asarray(rL, dtype=float)
    d = rL - rS
    d = d / np.linalg.norm(d)
    a = EQUATORIAL_RADIUS_KM
    b = POLAR_RADIUS_KM
    A = (d[0] ** 2 + d[1] ** 2) / a**2 + d[2] ** 2 / b**2
    B = 2 * ((rL[0] * d[0] + rL[1] * d[1]) / a**2 + rL[2] * d[2] / b**2)
    C = (rL[0] ** 2 + rL[1] ** 2) / a**2 + rL[2] ** 2 / b**2 - 1
    discriminant = B**2 - 4 * A * C
    if discriminant < 0:
        return (None, "no_central_line")
    root = np.sqrt(discriminant)
    solutions = [(-B - root) / (2 * A), (-B + root) / (2 * A)]
    solutions = [value for value in solutions if value >= 0]
    if not solutions:
        return (None, "no_central_line")
    point = rL + min(solutions) * d
    moon_distance = np.linalg.norm(point - rL)
    sun_distance = np.linalg.norm(point - rS)
    moon_angle = np.arcsin(np.clip(MOON_RADIUS_KM / moon_distance, 0, 1))
    sun_angle = np.arcsin(np.clip(SUN_RADIUS_KM / sun_distance, 0, 1))
    kind = "total" if moon_angle >= sun_angle else "annular"
    return (point, kind)

Convert the Cartesian intersection to geodetic longitude and latitude in degrees. Geodetic latitude follows the surface normal on the ellipsoid.

In [29]:
def lonlat(point):
    x, y, z = np.asarray(point, dtype=float)
    location = EarthLocation.from_geocentric(x, y, z, unit=u.km)
    lon, lat, _ = location.to_geodetic(ellipsoid="WGS84")
    return (float(lon.deg), float(lat.deg))

## Matching the orbital instant to Earth rotation

To displace a trajectory along the ground, it is enough to rotate Earth for the wrong amount of time. We use $\Delta T=\mathrm{TT}-\mathrm{UT1}$ in seconds. An event-specific value allows comparison with a NASA path produced under that assumption. A piecewise polynomial provides an estimate elsewhere.

In [30]:
def delta_t_polynomial(date_string):
    year, month, _ = date_parts(date_string)
    y = year + (month - 0.5) / 12
    if y < -500:
        u = (y - 1820) / 100
        dt = -20 + 32 * u**2
    elif y < 500:
        u = y / 100
        dt = (
            10583.6
            - 1014.41 * u
            + 33.78311 * u**2
            - 5.952053 * u**3
            - 0.1798452 * u**4
            + 0.022174192 * u**5
            + 0.0090316521 * u**6
        )
    elif y < 1600:
        u = (y - 1000) / 100
        dt = (
            1574.2
            - 556.01 * u
            + 71.23472 * u**2
            + 0.319781 * u**3
            - 0.8503463 * u**4
            - 0.005050998 * u**5
            + 0.0083572073 * u**6
        )
    elif y < 1700:
        t = y - 1600
        dt = 120 - 0.9808 * t - 0.01532 * t**2 + t**3 / 7129
    elif y < 1800:
        t = y - 1700
        dt = 8.83 + 0.1603 * t - 0.0059285 * t**2 + 0.00013336 * t**3 - t**4 / 1174000
    elif y < 1860:
        t = y - 1800
        dt = (
            13.72
            - 0.332447 * t
            + 0.0068612 * t**2
            + 0.0041116 * t**3
            - 0.00037436 * t**4
            + 1.21272e-05 * t**5
            - 1.699e-07 * t**6
            + 8.75e-10 * t**7
        )
    elif y < 1900:
        t = y - 1860
        dt = (
            7.62
            + 0.5737 * t
            - 0.251754 * t**2
            + 0.01680668 * t**3
            - 0.0004473624 * t**4
            + t**5 / 233174
        )
    elif y < 1920:
        t = y - 1900
        dt = (
            -2.79 + 1.494119 * t - 0.0598939 * t**2 + 0.0061966 * t**3 - 0.000197 * t**4
        )
    elif y < 1941:
        t = y - 1920
        dt = 21.2 + 0.84493 * t - 0.0761 * t**2 + 0.0020936 * t**3
    elif y < 1961:
        t = y - 1950
        dt = 29.07 + 0.407 * t - t**2 / 233 + t**3 / 2547
    elif y < 1986:
        t = y - 1975
        dt = 45.45 + 1.067 * t - t**2 / 260 - t**3 / 718
    elif y < 2005:
        t = y - 2000
        dt = (
            63.86
            + 0.3345 * t
            - 0.060374 * t**2
            + 0.0017275 * t**3
            + 0.000651814 * t**4
            + 2.373599e-05 * t**5
        )
    elif y < 2050:
        t = y - 2000
        dt = 62.92 + 0.32217 * t + 0.005589 * t**2
    elif y < 2150:
        dt = -20 + 32 * ((y - 1820) / 100) ** 2 - 0.5628 * (2150 - y)
    else:
        dt = -20 + 32 * ((y - 1820) / 100) ** 2
    if not 1955 <= y <= 2005:
        dt -= 1.2932e-05 * (y - 1955) ** 2
    return float(dt)

For dates in the path index, look for the event and read its adopted time difference. Record the source alongside the number so a comparison can distinguish a tabulated value from a polynomial estimate.

In [31]:
def get_delta_t(date_string, return_source=False):
    year, month, day = date_parts(date_string)
    source = "NASA Espenak-Meeus (polynomial estimate)"
    dt = None
    if 1991 <= year <= 2050:
        index = CACHE_DIR / "NASA_path_index.html"
        index_url = "https://eclipse.gsfc.nasa.gov/SEpath/SEpath.html"
        if index.exists():
            page = index.read_text(encoding="utf-8")
        else:
            with urlopen(index_url, timeout=30) as connection:
                page = connection.read().decode("utf-8")
            index.write_text(page, encoding="utf-8")
        event = f"SE{year}{MONTHS[month - 1]}{day:02d}"
        links = re.findall(
            "href=[\"\\']([^\"\\']+path\\.html)[\"\\']", page, flags=re.I
        )
        link = next((e for e in links if e.split("/")[-1].startswith(event)), None)
        if link is not None:
            file_path = CACHE_DIR / f"{event}_NASA_delta_t.html"
            if file_path.exists():
                html_text = file_path.read_text(encoding="utf-8")
            else:
                with urlopen(urljoin(index_url, link), timeout=30) as connection:
                    html_text = connection.read().decode("utf-8")
                file_path.write_text(html_text, encoding="utf-8")
            text = html.unescape(re.sub("<[^>]+>", " ", html_text))
            match = re.search("ΔT\\s*=\\s*([0-9]+(?:\\.[0-9]+)?)", text)
            if match is None:
                raise ValueError(f"NASA does not provide Delta T on page {event}.")
            dt = float(match.group(1))
            source = "NASA SEpath (adopted path value)"
    if dt is None:
        dt = delta_t_polynomial(date_string)
    if return_source:
        return (dt, source)
    return dt

At a specified UT1 midnight plus an elapsed interval, adding the adopted time difference gives the corresponding TT reading. We set the UT1−UTC offset consistently before constructing the terrestrial frame.

In [32]:
def make_time(date_string, seconds, delta_t):
    jd = start_ut1_day(date_string, delta_t)
    t = Time(jd, format="jd", scale="tt") + TimeDelta(
        float(seconds) + delta_t, format="sec"
    )
    tt_minus_utc = (t.tt.jd1 - t.utc.jd1 + (t.tt.jd2 - t.utc.jd2)) * 86400
    t.delta_ut1_utc = tt_minus_utc - delta_t
    return t

## Light reaching Earth

The classifier used Earth-centered vectors. Now we need to evaluate each body at a different instant, so Horizons supplies the barycentric positions and velocities of Earth, the Moon and the Sun every 300 s. Padding the interval supplies the earlier samples needed near midnight.

In [33]:
def fetch_physical_ephemerides(date_string, delta_t):
    label = cache_key({"delta_t": delta_t, "version": PIPELINE_VERSION})
    file_path = CACHE_DIR / f"JPL_barycentric_5min_TDB_UT1_{date_string}_DT{label}.csv"
    if file_path.exists():
        return pd.read_csv(file_path)
    start = make_time(date_string, 0, delta_t)
    begin = (start - TimeDelta(900, format="sec")).tdb.jd
    finish = (start + TimeDelta(86400 + 900, format="sec")).tdb.jd
    bodies = {"earth": "399", "moon": "301", "sun": "10"}
    df = None
    for name, body_id in bodies.items():
        params = {
            "format": "text",
            "COMMAND": f"'{body_id}'",
            "CENTER": "'500@0'",
            "MAKE_EPHEM": "'YES'",
            "OBJ_DATA": "'NO'",
            "EPHEM_TYPE": "'VECTORS'",
            "REF_SYSTEM": "'ICRF'",
            "REF_PLANE": "'FRAME'",
            "VEC_CORR": "'NONE'",
            "TIME_TYPE": "'TDB'",
            "OUT_UNITS": "'KM-S'",
            "VEC_TABLE": "'2'",
            "CSV_FORMAT": "'YES'",
            "VEC_LABELS": "'YES'",
            "START_TIME": f"'JD {begin:.9f}'",
            "STOP_TIME": f"'JD {finish:.9f}'",
            "STEP_SIZE": "'5 m'",
        }
        text_path = (
            CACHE_DIR
            / f"JPL_barycentric_5min_TDB_UT1_{date_string}_DT{label}_{name}.txt"
        )
        if text_path.exists():
            response = text_path.read_text(encoding="utf-8")
        else:
            url = "https://ssd.jpl.nasa.gov/api/horizons.api?" + urlencode(params)
            response = urlopen(url, timeout=60).read().decode("utf-8")
            if "$$SOE" not in response or "$$EOE" not in response:
                raise ValueError(f"Horizons error for {name}: {response[:300]}")
            text_path.write_text(response, encoding="utf-8")
        rows = list(
            csv.reader(
                io.StringIO(response.split("$$SOE")[1].split("$$EOE")[0].strip())
            )
        )
        data = np.array(
            [[float(row[i]) for i in (0, 2, 3, 4, 5, 6, 7)] for row in rows]
        )
        times = Time(data[:, 0], format="jd", scale="tdb")
        seconds = (times.tdb - start.tdb).to_value("s")
        if df is None:
            df = pd.DataFrame({"seconds": seconds})
        elif not np.allclose(df["seconds"], seconds, rtol=0, atol=0.002):
            raise ValueError("The three bodies have different sample times.")
        columns = [
            f"{variable}_{name}"
            for variable in ("x_km", "y_km", "z_km", "vx_kms", "vy_kms", "vz_kms")
        ]
        df[columns] = data[:, 1:]
    df.to_csv(file_path, index=False)
    return df

At reception time $t$ the light passed the Moon at $t_M=t-\tau_M$ and left the Sun at $t_S=t_M-\tau_{SM}$. For example, $c\tau_M=|\mathbf R_M(t_M)-\mathbf R_E(t)|$. Iterating that relation updates the travel time as the Moon moves. We repeat it on the Sun–Moon leg, subtract Earth's position at reception, then rotate into terrestrial axes. Cubic Hermite interpolation matches the sampled positions and velocities so a retarded instant can fall between ephemeris samples.

In [34]:
def transform_frame(date_string, df_jpl, delta_t):
    ephemerides = {}
    for name in ("earth", "moon", "sun"):
        position = df_jpl[[f"{e}_{name}" for e in ("x_km", "y_km", "z_km")]]
        velocity = df_jpl[[f"{e}_{name}" for e in ("vx_kms", "vy_kms", "vz_kms")]]
        ephemerides[name] = CubicHermiteSpline(
            df_jpl["seconds"],
            position.to_numpy(),
            velocity.to_numpy(),
            extrapolate=False,
        )
    earth = ephemerides["earth"]
    moon = ephemerides["moon"]
    sun = ephemerides["sun"]
    rows = []
    ut1_date = date_from_jd(start_ut1_day(date_string, delta_t))
    for s in np.arange(0, 86401, 300):
        RE = earth(s)
        tau_L = np.linalg.norm(moon(s) - RE) / LIGHT_SPEED_KM_S
        for _ in range(2):
            tau_L = np.linalg.norm(moon(s - tau_L) - RE) / LIGHT_SPEED_KM_S
        tL = s - tau_L
        tau_SL = np.linalg.norm(sun(tL) - moon(tL)) / LIGHT_SPEED_KM_S
        for _ in range(2):
            tau_SL = np.linalg.norm(sun(tL - tau_SL) - moon(tL)) / LIGHT_SPEED_KM_S
        tS = tL - tau_SL
        rL = moon(tL) - RE
        rS = sun(tS) - RE
        vL = moon.derivative()(tL) - earth.derivative()(s)
        vS = sun.derivative()(tS) - earth.derivative()(s)
        t = make_time(date_string, s, delta_t)
        rL_itrs, vL_itrs = rotate_state(t, rL, vL)
        rS_itrs, vS_itrs = rotate_state(t, rS, vS)
        row = {"seconds": s, "ut1": t.ut1.isot, "tau_L_s": tau_L, "tau_SL_s": tau_SL}
        for name, r, v in [("moon", rL_itrs, vL_itrs), ("sun", rS_itrs, vS_itrs)]:
            for coordinate, value in zip(("x", "y", "z"), r):
                row[f"{coordinate}_km_{name}"] = value
            for coordinate, value in zip(("vx", "vy", "vz"), v):
                row[f"{coordinate}_kms_{name}"] = value
        rows.append(row)
    df_itrs = pd.DataFrame(rows)
    interpolators = {}
    for name in ("moon", "sun"):
        r = df_itrs[[f"{e}_km_{name}" for e in ("x", "y", "z")]]
        v = df_itrs[[f"{e}_kms_{name}" for e in ("vx", "vy", "vz")]]
        interpolators[name] = CubicHermiteSpline(
            df_itrs["seconds"], r.to_numpy(), v.to_numpy(), extrapolate=False
        )
    return (df_itrs, interpolators)

The central intersections form a track in terrestrial coordinates. To obtain its ground velocity, evaluate the intersection on either side of the time of interest and divide the displacement by the elapsed seconds. Projection onto local east and north gives the two signed components.

In [35]:
def compute_geometry(df_itrs, interpolators):
    rows = []
    for s, ut1 in zip(df_itrs["seconds"], df_itrs["ut1"]):
        rS = interpolators["sun"](s)
        rL = interpolators["moon"](s)
        point, kind = central_point(rS, rL)
        row = {
            "seconds": s,
            "ut1": ut1,
            "kind": kind,
            "lat": np.nan,
            "lon": np.nan,
            "x": np.nan,
            "y": np.nan,
            "z": np.nan,
        }
        if point is not None:
            lon, lat = lonlat(point)
            row.update(lat=lat, lon=lon, x=point[0], y=point[1], z=point[2])
        rows.append(row)
    return pd.DataFrame(rows)

In [36]:
def compute_dynamics(geometry_df, interpolators, h=1.0):
    df = geometry_df.copy()
    df["east_speed_km_s"] = np.nan
    df["north_speed_km_s"] = np.nan
    df["speed_km_s"] = np.nan
    for i, row in df.dropna(subset=["lat", "lon"]).iterrows():
        s = row["seconds"]
        a, b = (
            max(interpolators["sun"].x[0], s - h),
            min(interpolators["sun"].x[-1], s + h),
        )
        previous, _ = central_point(interpolators["sun"](a), interpolators["moon"](a))
        following, _ = central_point(interpolators["sun"](b), interpolators["moon"](b))
        if previous is None or following is None:
            continue
        v = (following - previous) / (b - a)
        lon, lat = np.deg2rad([row["lon"], row["lat"]])
        east = np.array([-np.sin(lon), np.cos(lon), 0])
        north = np.array(
            [-np.sin(lat) * np.cos(lon), -np.sin(lat) * np.sin(lon), np.cos(lat)]
        )
        df.loc[i, "east_speed_km_s"] = np.dot(v, east)
        df.loc[i, "north_speed_km_s"] = np.dot(v, north)
        df.loc[i, "speed_km_s"] = np.linalg.norm(v)
    return df

A cached table contains values at the five-minute knots. Reconstructing the Hermite splines from positions and velocities restores the continuous interpolation used to calculate the shadow between those knots.

In [37]:
def restore_interpolators(df_itrs):
    interpolators = {}
    for name in ["sun", "moon"]:
        position = df_itrs[[f"{e}_km_{name}" for e in ["x", "y", "z"]]]
        velocity = df_itrs[[f"{e}_kms_{name}" for e in ["vx", "vy", "vz"]]]
        interpolators[name] = CubicHermiteSpline(
            df_itrs["seconds"],
            position.to_numpy(),
            velocity.to_numpy(),
            extrapolate=False,
        )
    return interpolators

## From an axis to a region

A single meridional section through the Sun and Moon contains two pairs of common tangents. Rotating each tangent about the Sun–Moon axis sweeps out a cone. To carry out this rotation we construct two perpendicular unit vectors transverse to the axis.

In [38]:
def cone_basis(rS, rL):
    axis = rL - rS
    distance = np.linalg.norm(axis)
    axis = axis / distance
    reference = np.array([0.0, 0.0, 1.0])
    if abs(axis[2]) > 0.9:
        reference = np.array([1.0, 0.0, 0.0])
    e1 = np.cross(axis, reference)
    e1 = e1 / np.linalg.norm(e1)
    e2 = np.cross(axis, e1)
    return (axis, e1, e2, distance)

We need both directions of the map transformation. Cone intersections start in Cartesian km and must become geographic points. For a grid search we start from geodetic longitude and latitude and reconstruct the Cartesian surface point on WGS84.

In [39]:
def contour_coordinates(points):
    location = EarthLocation.from_geocentric(
        points[:, 0], points[:, 1], points[:, 2], unit=u.km
    )
    lon, lat, _ = location.to_geodetic(ellipsoid="WGS84")
    return np.column_stack([lon.deg, lat.deg])

In [40]:
def wgs84_surface(longitudes, latitudes):
    lon = np.radians(longitudes)
    lat = np.radians(latitudes)
    e2 = FLATTENING * (2 - FLATTENING)
    N = EQUATORIAL_RADIUS_KM / np.sqrt(1 - e2 * np.sin(lat) ** 2)
    return np.column_stack(
        [
            N * np.cos(lat) * np.cos(lon),
            N * np.cos(lat) * np.sin(lon),
            N * (1 - e2) * np.sin(lat),
        ]
    )

## What does the observer actually see?

Take a point $P$ near sunrise. The solar and lunar disks can overlap while their common portion is still below the horizon. For a sightline $\hat{\mathbf u}$, the sign of $\hat{\mathbf n}\cdot\hat{\mathbf u}$ gives its position relative to the local horizon. We test the highest part of the overlapping disks, including their rim intersections when those exist.

In [41]:
def overlap_altitude(sun, moon, normal, rs, rl):
    best = np.full(len(sun), -1.0)
    for center, radius, other, other_radius in [
        (sun, rs, moon, rl),
        (moon, rl, sun, rs),
    ]:
        altitude = np.sum(normal * center, axis=1)
        tangent = normal - altitude[:, None] * center
        norm = np.linalg.norm(tangent, axis=1)
        tangent = tangent / np.maximum(norm[:, None], 1e-15)
        candidate = np.cos(radius)[:, None] * center + np.sin(radius)[:, None] * tangent
        inside = altitude >= np.cos(radius)
        candidate[inside] = normal[inside]
        valid = np.sum(candidate * other, axis=1) >= np.cos(other_radius)
        best[valid] = np.maximum(
            best[valid], np.sum(candidate[valid] * normal[valid], axis=1)
        )
    cross = np.cross(sun, moon)
    sine = np.linalg.norm(cross, axis=1)
    cosine = np.sum(sun * moon, axis=1)
    separation = np.arctan2(sine, cosine)
    intersect = (separation > abs(rs - rl)) & (separation < rs + rl)
    indices = np.flatnonzero(intersect & (sine > 1e-12))
    if len(indices):
        c = cosine[indices]
        denominator = sine[indices] ** 2
        A = (np.cos(rs[indices]) - c * np.cos(rl[indices])) / denominator
        B = (np.cos(rl[indices]) - c * np.cos(rs[indices])) / denominator
        base = A[:, None] * sun[indices] + B[:, None] * moon[indices]
        altitude = np.sqrt(np.maximum(0, 1 - np.sum(base * base, axis=1)))
        perpendicular = cross[indices] / sine[indices, None]
        value = np.sum(base * normal[indices], axis=1)
        value += altitude * abs(np.sum(perpendicular * normal[indices], axis=1))
        best[indices] = np.maximum(best[indices], value)
    return best

Let $\delta$ be the angular separation of the centers as seen from $P$, and $r_S,r_M$ their angular radii, all in radians. External contact gives $\delta=r_S+r_M$. For totality the Sun must fit inside the Moon, so $\delta\le r_M-r_S$. The code combines each overlap condition with visibility above the horizon. Only the **sign** of the combined quantity is meaningful. Its zero contour bounds the visible region.

In [42]:
def shadow_margin(points, rS, rL, kind, horizon=True):
    sun = rS - points
    moon = rL - points
    ds = np.linalg.norm(sun, axis=1)
    dl = np.linalg.norm(moon, axis=1)
    sun = sun / ds[:, None]
    moon = moon / dl[:, None]
    rs = np.arcsin(SUN_RADIUS_KM / ds)
    rl = np.arcsin(MOON_RADIUS_KM / dl)
    separation = np.arctan2(
        np.linalg.norm(np.cross(sun, moon), axis=1), np.sum(sun * moon, axis=1)
    )
    margin = rs + rl - separation
    if kind == "umbra":
        margin = rl - rs - separation
    if not horizon:
        return margin
    semiaxes = np.array([EQUATORIAL_RADIUS_KM] * 2 + [POLAR_RADIUS_KM])
    normal = points / semiaxes**2
    normal = normal / np.linalg.norm(normal, axis=1)[:, None]
    altitude = np.sum(normal * sun, axis=1)
    upper_edge = altitude * np.cos(rs) + np.sqrt(
        np.maximum(0, 1 - altitude**2)
    ) * np.sin(rs)
    margin = np.minimum(margin, upper_edge)
    if kind == "penumbra":
        indices = np.flatnonzero((margin > 0) & (altitude < np.sin(rs)))
        if len(indices):
            visible = overlap_altitude(
                sun[indices], moon[indices], normal[indices], rs[indices], rl[indices]
            )
            margin[indices] = np.minimum(margin[indices], visible)
    return margin

For center distance $D$, the tangent geometry gives $\sin\phi=(R_S+R_M)/D$ for the expanding penumbral cone and $(R_S-R_M)/D$ for the converging umbral cone. We sample azimuths around the axis and intersect each generator with the ellipsoid. On the umbral branch we also require that the intersection lie before the vertex. Beyond it the apparent Moon is too small for totality.

In [43]:
def cone_points(rS, rL, kind, point_count):
    axis, e1, e2, distance = cone_basis(rS, rL)
    sign = 1 if kind == "penumbra" else -1
    sine = (SUN_RADIUS_KM + sign * MOON_RADIUS_KM) / distance
    cosine = np.sqrt(1 - sine**2)
    theta = np.arange(point_count) * 2 * np.pi / point_count
    radial = np.cos(theta)[:, None] * e1 + np.sin(theta)[:, None] * e2
    origin = rL + MOON_RADIUS_KM * (cosine * radial - sign * sine * axis)
    direction = cosine * axis + sign * sine * radial
    semiaxes = np.array([EQUATORIAL_RADIUS_KM] * 2 + [POLAR_RADIUS_KM])
    p = origin / semiaxes
    d = direction / semiaxes
    A = np.sum(d * d, axis=1)
    B = 2 * np.sum(p * d, axis=1)
    C = np.sum(p * p, axis=1) - 1
    discriminant = B * B - 4 * A * C
    root = np.sqrt(np.maximum(0, discriminant))
    t = (-B - root) / (2 * A)
    points = origin + t[:, None] * direction
    valid = (discriminant >= 0) & (t >= 0)
    if kind == "umbra":
        valid &= (points - rL) @ axis < MOON_RADIUS_KM / sine
    return (points, valid)

If some generators miss Earth, joining the remaining points directly may draw a chord across the wrong region. We also sample the solar horizon as a guide for locating the open ends of the contour.

In [44]:
def horizon_points(rS, point_count=1440):
    semiaxes = np.array([EQUATORIAL_RADIUS_KM] * 2 + [POLAR_RADIUS_KM])
    v = rS / semiaxes
    distance = np.linalg.norm(v)
    axis = v / distance
    _, e1, e2, _ = cone_basis(np.zeros(3), axis)
    theta = np.arange(point_count) * 2 * np.pi / point_count
    radial = np.cos(theta)[:, None] * e1 + np.sin(theta)[:, None] * e2
    esfera = axis / distance + np.sqrt(1 - 1 / distance**2) * radial
    return esfera * semiaxes

## Drawing the boundary in longitude and latitude

A region crossing 180° longitude is continuous on Earth but occupies both edges of a flat map. We keep polygon pieces from the geometric operations, cut them into longitude strips and shift them back into the displayed range.

In [45]:
def polygon_components(compute_geometry):
    if compute_geometry.is_empty:
        return []
    if compute_geometry.geom_type == "Polygon":
        return [compute_geometry]
    parts = []
    if hasattr(compute_geometry, "geoms"):
        for component in compute_geometry.geoms:
            parts.extend(polygon_components(component))
    return parts

In [46]:
def split_antimeridian(polygons):
    result = []
    for polygon in polygons:
        if not polygon.is_valid:
            raise ValueError(
                "Invalid polygon: increase resolution and inspect the boundary."
            )
        minimum, _, maximum, _ = polygon.bounds
        start = int(np.floor((minimum + 180) / 360))
        end = int(np.floor((maximum + 180) / 360))
        for turn in range(start, end + 1):
            window = box(-180 + 360 * turn, -90, 180 + 360 * turn, 90)
            parts = polygon_components(polygon.intersection(window))
            for part in parts:
                result.append(translate(part, xoff=-360 * turn))
    return result

The grid locates edges across which the margin changes sign. Bisection along one such edge moves the vertex toward the physical zero of the margin. This refines a crossing while retaining the connectivity found by the grid.

In [47]:
def refine_boundary(ring, x, y, rS, rL, kind):
    dx, dy = (x[1] - x[0], y[1] - y[0])
    ix = np.clip(np.floor((ring[:, 0] - x[0]) / dx).astype(int), 0, len(x) - 2)
    iy = np.clip(np.floor((ring[:, 1] - y[0]) / dy).astype(int), 0, len(y) - 2)
    near_x = np.clip(np.rint((ring[:, 0] - x[0]) / dx).astype(int), 0, len(x) - 1)
    vertical = abs(ring[:, 0] - x[near_x]) < 1e-08
    a = ring.copy()
    b = ring.copy()
    a[vertical, 1] = y[iy[vertical]]
    b[vertical, 1] = y[iy[vertical] + 1]
    a[~vertical, 0] = x[ix[~vertical]]
    b[~vertical, 0] = x[ix[~vertical] + 1]
    fa = shadow_margin(wgs84_surface(a[:, 0], a[:, 1]), rS, rL, kind)
    fb = shadow_margin(wgs84_surface(b[:, 0], b[:, 1]), rS, rL, kind)
    change = fa * fb < 0
    limit = (abs(ring[:, 0] - x[0]) < 1e-08) | (abs(ring[:, 0] - x[-1]) < 1e-08)
    limit |= (abs(ring[:, 1] - y[0]) < 1e-08) | (abs(ring[:, 1] - y[-1]) < 1e-08)
    change &= ~limit
    indices = np.flatnonzero(change)
    left, right, value = (a[indices], b[indices], fa[indices])
    for _ in range(16):
        middle = (left + right) / 2
        fm = shadow_margin(wgs84_surface(middle[:, 0], middle[:, 1]), rS, rL, kind)
        same_sign = value * fm > 0
        left[same_sign] = middle[same_sign]
        value[same_sign] = fm[same_sign]
        right[~same_sign] = middle[~same_sign]
    result = ring.copy()
    result[indices] = (left + right) / 2
    result[-1] = result[0]
    return result

When every generator meets Earth and the ring stays clear of the horizon and polar wrapping, its intersections already define a closed boundary. Otherwise we sample the visibility margin on a local grid. The zero contour can then follow both disk contact and the horizon through the same construction.

In [48]:
def shadow_with_horizon(rS, rL, kind, point_count=720, grid_step=0.25):
    points, valid = cone_points(rS, rL, kind, point_count)
    reference = np.degrees(np.arctan2(rL[1], rL[0]))
    candidates = points[valid]
    if valid.all():
        coords = contour_coordinates(points)
        lon = np.degrees(np.unwrap(np.radians(np.r_[coords[:, 0], coords[0, 0]])))
        lat = np.r_[coords[:, 1], coords[0, 1]]
        semiaxes = np.array([EQUATORIAL_RADIUS_KM] * 2 + [POLAR_RADIUS_KM])
        normal = points / semiaxes**2
        normal /= np.linalg.norm(normal, axis=1)[:, None]
        sun = rS - points
        sun /= np.linalg.norm(sun, axis=1)[:, None]
        if abs(lon[-1] - lon[0]) < 180 and np.min(np.sum(sun * normal, axis=1)) > 0.02:
            polygon = Polygon(np.column_stack([lon, lat]))
            return (split_antimeridian([polygon]), "cone_rays")
    edge = horizon_points(rS)
    inside = shadow_margin(edge, rS, rL, kind, horizon=False) >= 0
    candidates = np.concatenate([candidates, edge[inside]])
    poles = wgs84_surface(np.array([0, 0]), np.array([-90, 90]))
    poles_inside = shadow_margin(poles, rS, rL, kind) > 0
    if not len(candidates) and (not poles_inside.any()):
        return ([], "no_sampled_region")
    if len(candidates):
        coords = contour_coordinates(candidates)
        lon = reference + (coords[:, 0] - reference + 180) % 360 - 180
        west, east = (lon.min() - 1, lon.max() + 1)
        south, north = (coords[:, 1].min() - 1, coords[:, 1].max() + 1)
    else:
        west, east, south, north = (reference - 180, reference + 180, -90, 90)
    if poles_inside.any():
        west, east = (reference - 180, reference + 180)
        if poles_inside[0]:
            south = -90
        if poles_inside[1]:
            north = 90
    for attempt in range(8):
        west = max(west, reference - 180)
        east = min(east, reference + 180)
        south, north = (max(south, -90), min(north, 90))
        x = np.linspace(
            west, east, max(25, int(np.ceil((east - west) / grid_step)) + 1)
        )
        y = np.linspace(
            south, north, max(25, int(np.ceil((north - south) / grid_step)) + 1)
        )
        z = np.empty((len(y), len(x)))
        for start in range(0, len(y), 40):
            xx, yy = np.meshgrid(x, y[start : start + 40])
            ground = wgs84_surface(xx.ravel(), yy.ravel())
            values = shadow_margin(ground, rS, rL, kind)
            z[start : start + 40] = values.reshape(xx.shape)
        exceeds_west = np.any(z[:, 0] > 0) and west > reference - 180
        exceeds_east = np.any(z[:, -1] > 0) and east < reference + 180
        exceeds_south = np.any(z[0] > 0) and south > -90
        exceeds_north = np.any(z[-1] > 0) and north < 90
        if not any([exceeds_west, exceeds_east, exceeds_south, exceeds_north]):
            break
        margin = 2 ** (attempt + 1)
        if exceeds_west:
            west -= margin
        if exceeds_east:
            east += margin
        if exceeds_south:
            south -= margin
        if exceeds_north:
            north += margin
    else:
        raise ValueError("The shadow reaches the expanded grid limits.")
    generator = contourpy.contour_generator(x=x, y=y, z=z, fill_type="OuterOffset")
    vertices, offsets = generator.filled(0, np.inf)
    polygons = []
    for region_points, cuts in zip(vertices, offsets):
        rings = []
        for j in range(len(cuts) - 1):
            ring = region_points[cuts[j] : cuts[j + 1]]
            rings.append(refine_boundary(ring, x, y, rS, rL, kind))
        polygons.append(Polygon(rings[0], rings[1:]))
    return (split_antimeridian(polygons), "horizon_grid")

## How long should we follow the shadow?

A conservative test compares the axis distance from Earth with the Earth radius plus a bound on the penumbral radius. We screen at one-minute intervals and pad the candidate interval by a minute at either end. This reduces the number of expensive surface contours we need to calculate.

In [49]:
def possible_penumbra(rS, rL):
    axis, _, _, distance = cone_basis(rS, rL)
    axial = np.dot(-rL, axis)
    perpendicular = np.linalg.norm(-rL - axial * axis)
    sine = (SUN_RADIUS_KM + MOON_RADIUS_KM) / distance
    tangent = sine / np.sqrt(1 - sine * sine)
    radius = MOON_RADIUS_KM / np.sqrt(1 - sine * sine)
    radius += (axial + EQUATORIAL_RADIUS_KM) * tangent
    return axial > 0 and perpendicular <= EQUATORIAL_RADIUS_KM + radius

In [50]:
def eclipse_interval(interpolators):
    start = max(interpolators["sun"].x[0], interpolators["moon"].x[0])
    end = min(interpolators["sun"].x[-1], interpolators["moon"].x[-1])
    times = np.r_[np.arange(start, end, 60), end]
    possible = []
    for s in times:
        if possible_penumbra(interpolators["sun"](s), interpolators["moon"](s)):
            possible.append(s)
    if not possible:
        return None
    if possible[0] == start or possible[-1] == end:
        raise ValueError("The event extends beyond the available ephemerides.")
    return (max(start, possible[0] - 60), min(end, possible[-1] + 60))

For each sampled instant $t_i$ we now have a surface region $S(t_i)$. The map displays $\bigcup_i S(t_i)$, the places reached at least once during the sampled event. Two cities inside that region can see the eclipse at different times. The default spacing is 10 s with 720 generators. Smaller time steps can reduce gaps between snapshots. A finer angular grid alone cannot fill such gaps.

In [51]:
def sweep_shadows(
    interpolators,
    time_step_s=10,
    point_count=720,
    penumbra_grid_deg=0.25,
    umbra_grid_deg=0.05,
    show_progress=True,
):
    if (
        time_step_s <= 0
        or penumbra_grid_deg <= 0
        or umbra_grid_deg <= 0
        or (point_count < 36)
    ):
        raise ValueError("Steps must be positive and point_count must be at least 36.")
    interval = eclipse_interval(interpolators)
    accumulated = {"penumbra": [], "umbra": []}
    rows, central_points = ([], [])
    if interval is None:
        return {
            "penumbra": Polygon(),
            "umbra": Polygon(),
            "samples": pd.DataFrame(),
            "central_points": [],
            "interval": None,
        }
    start, end = interval
    times = np.r_[np.arange(start, end, time_step_s), end]
    for index, s in enumerate(times):
        rS, rL = (interpolators["sun"](s), interpolators["moon"](s))
        row = {"seconds": float(s)}
        for kind, step in [("penumbra", penumbra_grid_deg), ("umbra", umbra_grid_deg)]:
            polygons, method = shadow_with_horizon(rS, rL, kind, point_count, step)
            accumulated[kind].extend(polygons)
            row[kind] = bool(polygons)
            row["method_" + kind] = method
            if index % 100 == 99 and accumulated[kind]:
                accumulated[kind] = [unary_union(accumulated[kind])]
        point, central_kind = central_point(rS, rL)
        if point is None:
            central_points.append(None)
        else:
            central_points.append(lonlat(point))
        rows.append(row)
        if show_progress and index % 100 == 0:
            print(f"Sample {index + 1}/{len(times)}", flush=True)
    result = {
        "interval": interval,
        "samples": pd.DataFrame(rows),
        "central_points": central_points,
        "time_step_s": time_step_s,
    }
    for kind in ["penumbra", "umbra"]:
        result[kind] = unary_union(accumulated[kind])
        if not result[kind].is_valid:
            raise ValueError("The union produced an invalid geometry.")
    return result

A central intersection can disappear, or its plotted longitude can jump from one edge of the map to the other. Break the plotted line at either gap so the map does not connect those points across the world.

In [52]:
def central_line_segments(coordinates):
    segments, current = ([], [])
    for point in coordinates:
        split = point is None
        if point is not None and current:
            split = abs(point[0] - current[-1][0]) > 180
        if split:
            if len(current) >= 2:
                segments.append(current)
            current = []
        if point is not None:
            current.append(point)
    if len(current) >= 2:
        segments.append(current)
    return segments

## Putting the days together

For a date and adopted time difference we can reuse the transformed terrestrial states. If the penumbra may intersect Earth at midnight, we add the neighboring day and express all timestamps in seconds from the same origin.

In [53]:
def get_physical_day(date_string, delta_t):
    key = cache_key(
        {"date": date_string, "delta_t_s": delta_t, "version": PIPELINE_VERSION}
    )
    path = CACHE_DIR / ("itrs_" + key + ".csv")
    if path.exists():
        table = pd.read_csv(path)
        return (table, restore_interpolators(table))
    ephemerides = fetch_physical_ephemerides(date_string, delta_t)
    table, interpolators = transform_frame(date_string, ephemerides, delta_t)
    table.to_csv(path, index=False)
    return (table, interpolators)

In [54]:
def prepare_path(date_string, delta_t):
    table, interpolators = get_physical_day(date_string, delta_t)
    tables = [table]
    origin = make_time(date_string, 0, delta_t)
    for endpoint, offset in [(0, -1), (86400, 1)]:
        if possible_penumbra(
            interpolators["sun"](endpoint), interpolators["moon"](endpoint)
        ):
            adjacent_date = date_from_jd(julian_day(date_string) + offset)
            date_parts(adjacent_date)
            adjacent_table, _ = get_physical_day(adjacent_date, delta_t)
            shift = (make_time(adjacent_date, 0, delta_t).tt - origin.tt).to_value("s")
            adjacent_table = adjacent_table.copy()
            adjacent_table["seconds"] += shift
            tables.append(adjacent_table)
    combined = pd.concat(tables).sort_values("seconds").drop_duplicates("seconds")
    return (combined, restore_interpolators(combined))

The classifier now determines whether to compute the path. For positive dates we collect the physical line, its velocity and both swept regions. Model 2 reduced the recorded mean test displacement by about 56 m, with some eclipses becoming less accurate. We keep the geometric trajectory here. The final coordinate cache includes the temporal and angular sampling settings.

In [55]:
def run_eclipse(
    date_string,
    time_step_s=10,
    point_count=720,
    penumbra_grid_deg=0.25,
    umbra_grid_deg=0.05,
    rebuild_coordinates=False,
):
    date_string = normalize_date(date_string)
    prediction = predict_eclipse(date_string)
    result = {
        "date": date_string,
        "prediction": prediction,
        "has_eclipse": prediction["prediction"],
    }
    if not result["has_eclipse"]:
        return result
    settings = {
        "date": date_string,
        "model_id": prediction["model_id"],
        "version": PIPELINE_VERSION,
        "time_step_s": time_step_s,
        "point_count": point_count,
        "penumbra_grid_deg": penumbra_grid_deg,
        "umbra_grid_deg": umbra_grid_deg,
    }
    path = CACHE_DIR / ("coordinates_" + cache_key(settings) + ".json")
    if path.exists() and (not rebuild_coordinates):
        result = json.loads(path.read_text())
        result["coordinate_cache_hit"] = True
        return result
    delta_t, source = get_delta_t(date_string, return_source=True)
    itrs, interpolators = prepare_path(date_string, delta_t)
    geometry = compute_geometry(itrs, interpolators)
    dynamics = compute_dynamics(geometry, interpolators)
    sweep = sweep_shadows(
        interpolators,
        time_step_s,
        point_count,
        penumbra_grid_deg,
        umbra_grid_deg,
        show_progress=True,
    )
    result.update(
        {
            "settings": settings,
            "delta_t_s": delta_t,
            "delta_t_source": source,
            "coordinate_cache_hit": False,
            "interval_s": sweep["interval"],
            "penumbra": mapping(sweep["penumbra"]),
            "umbra": mapping(sweep["umbra"]),
            "central_line": central_line_segments(sweep["central_points"]),
            "dynamics": json.loads(dynamics.to_json(orient="records")),
            "samples": json.loads(sweep["samples"].to_json(orient="records")),
            "physical_region_found": not sweep["penumbra"].is_empty,
        }
    )
    save_json(path, result)
    return result

## Reading the map

The outer layer includes every sampled point with visible disk overlap, including points that also reached totality. The umbra marks totality. An annular event can have a central line and an empty umbral layer. There is no separate annularity polygon.

In [56]:
def create_map(result):
    if not result["has_eclipse"]:
        raise ValueError("Model 1 predicted no eclipse; no physical map was computed.")
    eclipse_map = folium.Map(
        location=[15, 0], zoom_start=2, tiles="OpenStreetMap", control_scale=True
    )
    for kind, color, opacity, order in [
        ("penumbra", "#777777", 0.3, 410),
        ("umbra", "#151515", 0.8, 420),
    ]:
        folium.map.CustomPane(kind, z_index=order).add_to(eclipse_map)
        layer = folium.FeatureGroup(name=kind.capitalize())
        for polygon in polygon_components(shape(result[kind])):
            rings = [
                [[lat, lon] for lon, lat in ring.coords]
                for ring in [polygon.exterior] + list(polygon.interiors)
            ]
            folium.Polygon(
                rings,
                color=color,
                weight=1,
                fill=True,
                fill_color=color,
                fill_opacity=opacity,
                pane=kind,
                smooth_factor=0,
                tooltip=kind.capitalize() + ": covered at some time",
            ).add_to(layer)
        layer.add_to(eclipse_map)
    folium.map.CustomPane("central", z_index=430).add_to(eclipse_map)
    layer = folium.FeatureGroup(name="Central line")
    for segment in result["central_line"]:
        folium.PolyLine(
            [[lat, lon] for lon, lat in segment],
            color="#e86432",
            weight=2,
            pane="central",
            smooth_factor=0,
        ).add_to(layer)
    layer.add_to(eclipse_map)
    region = shape(result["penumbra"])
    if not region.is_empty:
        west, south, east, north = region.bounds
        eclipse_map.fit_bounds([[max(south, -85), west], [min(north, 85), east]])
    folium.LayerControl(collapsed=False).add_to(eclipse_map)
    return eclipse_map

To calculate a date directly in Jupyter, enable the example below. It uses the same entry point as the dashboard.

In [57]:
RUN_EXAMPLE = False
example_date = "2024-04-08"
if RUN_EXAMPLE:
    result = run_eclipse(example_date)
    if result["has_eclipse"]:
        display(create_map(result))
    else:
        print("Model 1 predicts no eclipse for this date.")

## Using the same calculation in Streamlit

The dashboard runs in another Python process. We write the cells tagged `engine` and the embedded training table into an importable module, so the app evaluates exactly the functions developed above. Rerun this export after changing a physical or classification function.

In [58]:
notebook_document = json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))
engine_parts = [
    "".join(cell["source"])
    for cell in notebook_document["cells"]
    if cell["cell_type"] == "code"
    and "engine" in cell.get("metadata", {}).get("tags", [])
]
engine_parts.append("TRAINING_DATA = " + repr(TRAINING_DATA))
engine_path = PROJECT_DIR / "eclipse_engine.py"
engine_path.write_text("\n\n".join(engine_parts), encoding="utf-8")
print("Engine created:", engine_path)

Engine created: C:\Users\andre\Desktop\Eclipse_Project\eclipse_lab\eclipse_engine.py


The interface retains the selected date and result across reruns. A negative prediction stops at the result screen. A positive prediction opens the computed layers, and a failed request is reported as an error. An optional `eclipse_lab/assets/background.jpg` supplies the background image.

In [59]:
%%writefile eclipse_lab/app.py
import base64
from datetime import date
from pathlib import Path
import streamlit as st
from eclipse_engine import run_eclipse, create_map, PROJECT_DIR

st.set_page_config(page_title="Eclipse Lab", page_icon="🌘", layout="wide")


def set_background():
    path = PROJECT_DIR / "assets" / "background.jpg"
    background = "linear-gradient(135deg, #030711, #102643)"
    if path.exists():
        image = base64.b64encode(path.read_bytes()).decode()
        background = f'url("data:image/jpeg;base64,{image}")'
    st.markdown(
        f"""<style>
    .stApp {{background-image:{background};background-size:cover;background-position:center;}}
    .block-container {{max-width:1000px;padding-top:4rem;}}
    [data-testid="stHeader"] {{background:transparent;}}
    .center {{text-align:center;margin:4rem 0 2rem;color:white;}}
    .title {{font-size:3.5rem;font-weight:bold;}}
    .subtitle {{font-size:1.1rem;color:#bdd0e7;}}
    .date {{font-size:3.8rem;font-weight:bold;}}
    </style>""",
        unsafe_allow_html=True,
    )


set_background()
st.markdown("**🌘 ECLIPSE LAB**")
if "screen" not in st.session_state:
    st.session_state.screen = "home"

if st.session_state.screen == "home":
    st.markdown(
        """<div class="center"><div class="title">Explore a solar eclipse</div>
    <div class="subtitle">Choose a date and discover where its shadow travels.</div></div>""",
        unsafe_allow_html=True,
    )
    left, center, right = st.columns([1, 2, 1])
    with center:
        selected_date = st.date_input(
            "Select a date",
            value=date(2024, 4, 8),
            min_value=date(1, 1, 1),
            max_value=date(3000, 12, 31),
            format="YYYY-MM-DD",
        )
        date_string = selected_date.isoformat()
        if st.checkbox("Use astronomical year input (including BCE)"):
            date_string = st.text_input(
                "Gregorian date: YYYY-MM-DD", value="-0001-06-01"
            )
        if st.button("Find eclipse", use_container_width=True):
            try:
                with st.spinner(
                    "Analyzing the date and calculating the eclipse path..."
                ):
                    result = run_eclipse(date_string)
                st.session_state.result = result
                st.session_state.screen = "result"
                st.rerun()
            except Exception as error:
                st.error(f"The calculation could not be completed: {error}")
                st.info(
                    "Check your internet connection. Completed cache files will be reused when you retry."
                )

elif st.session_state.screen == "result":
    result = st.session_state.result
    label = (
        "Model 1 predicts a solar eclipse"
        if result["has_eclipse"]
        else "Model 1 predicts no solar eclipse"
    )
    st.markdown(
        f"""<div class="center"><div class="subtitle">{label}</div>
    <div class="date">{result['date']}</div></div>""",
        unsafe_allow_html=True,
    )
    st.caption(
        f"Model score: {result['prediction']['probability']:.1%}. This is a classifier output, not a confirmed catalog entry."
    )
    if result["prediction"]["outside_training_range"]:
        st.warning(
            "This date is outside the training period. The classifier is extrapolating."
        )
    if result["has_eclipse"]:
        if not result["physical_region_found"]:
            st.warning(
                "The physical calculation found no shadow region at the selected resolution."
            )
        if st.button("View eclipse path", use_container_width=True):
            st.session_state.screen = "map"
            st.rerun()
    if st.button("Search another date"):
        st.session_state.screen = "home"
        st.rerun()

elif st.session_state.screen == "map":
    result = st.session_state.result
    st.title("Eclipse path")
    st.write("Date:", result["date"])
    st.info(
        "Gray: penumbra. Dark: umbra (totality). Orange: central line. The map combines the entire event, not one instant."
    )
    map_html = create_map(result).get_root().render()
    if hasattr(st, "iframe"):
        st.iframe(map_html, height=580)
    else:
        st.components.v1.html(map_html, height=580)
    st.caption(
        "An annular eclipse has no umbra on Earth; its annular path is not a totality region. Polar regions above about 85° are clipped by the basemap."
    )
    if st.button("Back to result"):
        st.session_state.screen = "result"
        st.rerun()
    if st.button("Search another date"):
        st.session_state.screen = "home"
        st.rerun()

Writing eclipse_lab/app.py


Run the following cell to start the server locally. Open the displayed address while the kernel is running.

In [60]:
import subprocess
import sys
import time
import socket
from urllib.request import urlopen
from IPython.display import display, Markdown


def start_dashboard(port=8501):
    global dashboard_process
    if "dashboard_process" in globals() and dashboard_process.poll() is None:
        return f"http://127.0.0.1:{port}"
    with socket.socket() as probe:
        if probe.connect_ex(("127.0.0.1", port)) == 0:
            raise RuntimeError(
                f"Port {port} is already in use. Stop that server or choose another port."
            )
    log_path = PROJECT_DIR / "streamlit.log"
    with log_path.open("w", encoding="utf-8") as log:
        dashboard_process = subprocess.Popen(
            [
                sys.executable,
                "-m",
                "streamlit",
                "run",
                str(PROJECT_DIR / "app.py"),
                "--server.address",
                "127.0.0.1",
                "--server.port",
                str(port),
                "--server.headless",
                "true",
            ],
            cwd=str(PROJECT_DIR.parent),
            stdout=log,
            stderr=subprocess.STDOUT,
        )
    url = f"http://127.0.0.1:{port}"
    for attempt in range(60):
        if dashboard_process.poll() is not None:
            break
        try:
            with urlopen(url + "/_stcore/health", timeout=1) as response:
                if response.status == 200:
                    return url
        except OSError:
            time.sleep(0.5)
    raise RuntimeError(
        "Dashboard startup failed. " + log_path.read_text(encoding="utf-8")[-3000:]
    )


dashboard_url = start_dashboard()
display(Markdown(f"[Open Eclipse Lab]({dashboard_url})"))

[Open Eclipse Lab](http://127.0.0.1:8501)

Set the stop flag below when the session is finished.

In [61]:
STOP_DASHBOARD = False
if (
    STOP_DASHBOARD
    and "dashboard_process" in globals()
    and dashboard_process.poll() is None
):
    dashboard_process.terminate()
    try:
        dashboard_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        dashboard_process.kill()
        dashboard_process.wait()
    print("Dashboard stopped.")

## Interpreting a point on the finished map

In the original 8 April 2024 run, the default sweep used 1,873 snapshots and the adopted NASA value was 70.6 s. Dallas fell in both layers, San José in Costa Rica only in the penumbra, and Quito outside both. These three checks answer the opening question for specific places. Quantifying an error along the full shadow boundary would require a separate comparison.

The returned `samples` table identifies which contour construction was used at each time. The `dynamics` records give central positions and ground speeds. GeoJSON coordinates are ordered longitude, latitude in degrees. The map is a sampled envelope, and its Web Mercator basemap omits the immediate polar caps.